# shelf-vision: full detector training on Kaggle

Trains a YOLO11 detector on SKU-110K (`CONFIG` in the first code cell picks
`configs/detect/<CONFIG>.yaml`) for as long as one Kaggle session allows, then saves
the weights to the notebook output.

**Before running:** Settings → Accelerator **GPU T4 x2** (P100 also works),
Internet **on**. Then **Save Version → Save & Run All (Commit)** so it runs in the
background and the output is kept.

The run uses `shelf train --hours`: after the first epoch, ultralytics fits the
epoch count and learning-rate schedule into the remaining time, so the schedule
finishes properly instead of being cut off by the 12-hour limit.

In [ ]:
import os
import shutil
import time

CONFIG = "yolo11m-sku110k"  # configs/detect/<CONFIG>.yaml
CFG = f"shelf-vision/configs/detect/{CONFIG}.yaml"

SESSION_START = time.time()
SESSION_HOURS = 12.0  # Kaggle limit for a GPU session
FINISH_MARGIN = 0.75  # hours kept for final validation, saving output, slow epochs

DATA_DIR = "/tmp/shelf-data"  # ~24 GB peak; /kaggle/working is limited to ~20 GB
os.environ["SHELF_DATA_DIR"] = DATA_DIR

!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /tmp /kaggle/working | sed 1d
free_gb = shutil.disk_usage("/tmp").free / 2**30
assert free_gb > 30, f"only {free_gb:.0f} GB free in /tmp, need ~30 GB for the dataset"

In [ ]:
!git clone -q https://github.com/rsuvbe/shelf-vision
!git -C shelf-vision log --oneline -1
!pip install -q -e shelf-vision
!shelf doctor

## Data

Parallel download of the 11.6 GB archive (the S3 bucket is slow on a single
connection), then extraction and label conversion. The archive is deleted
afterwards to free disk space.

In [ ]:
!bash shelf-vision/scripts/fetch_sku110k.sh {DATA_DIR}
!shelf prepare {CFG}
!rm -f {DATA_DIR}/SKU110K_fixed.tar.gz
!du -sh {DATA_DIR}
print(f"setup took {(time.time() - SESSION_START) / 60:.0f} min")

## Train

Uses whatever time is left in the session. Batch size is scaled to the GPUs
(`batch_per_8gb` in the config); with 2 GPUs ultralytics runs DDP.

In [ ]:
elapsed_h = (time.time() - SESSION_START) / 3600
hours = round(SESSION_HOURS - elapsed_h - FINISH_MARGIN, 2)
print(f"training for {hours} h")
!shelf train {CFG} --hours {hours} 2>&1 | tr '\r' '\n' | grep -v '━' | tee /kaggle/working/train.log

## Test set

Held-out test split (2,936 photos), never seen during training or model selection.

In [ ]:
import glob

best = max(glob.glob(f"/kaggle/working/runs/detect/{CONFIG}-*h/weights/best.pt"))
print(best)
!shelf evaluate {best} {CFG} --split test 2>&1 | tr '\r' '\n' | grep -v '━' | tail -5 | tee -a /kaggle/working/train.log

Output to download: `runs/detect/<run>/weights/best.pt` and `last.pt`, plus
`results.csv`, plots and `train.log`. Put `best.pt` locally under the same
`runs/detect/<run>/weights/` path for evaluation and export.